## Preprocess the IEDB peptide-HLA complex data

In [2]:
import pandas as pd
import numpy as np
import os 
import sys
from utils import *
import peptides
peptides.__version__

'0.5.0'

In [4]:
# Feature processing for sequences, we will use the peptides package to extract features from the sequences. We will process the sequences in batches to avoid memory issues.
#Parse all the sequences and convert them into feature vectors
def process_batch(seqs_batch):
    return [extract_features(s) for s in seqs_batch]

def feature_processing(seqs, seq_type):
    batch_size = 5000
    all_features = []
    for i in range(0, len(seqs), batch_size):
        print("Finished batch "+str(i))
        batch = seqs[i:i + batch_size]
        batch_features = process_batch(batch)
        all_features.extend(batch_features)
    
    all_features_df = pd.DataFrame(all_features)
    all_features_df.columns = [seq_type+x for x in all_features_df.columns]
    return(all_features_df) 

In [5]:
#Load the unprocessed train data 
unprocessed_train_df = pd.read_csv("../data/IEDB_pHLA_data.csv",header="infer")
unprocessed_train_df.head(5)

,peptide,HLA,immunogenicity,test,respond,potential
0,KEHVFFSEY,HLA-B*4402,Negative,4,0,0.347444
1,DEGATLYRF,HLA-B*4402,Negative,4,0,0.346545
2,TLAARIKFL,HLA-A*0201,Negative,4,0,0.346239
3,KETLNEYKQL,HLA-B*4402,Negative,4,0,0.345162
4,STTDAEACY,HLA-A*0101,Negative,4,0,0.343674


In [6]:
#Convert negative and positive labels to 0 and 1 and keep as a separate column in dataframe
unprocessed_train_df["Label"]=unprocessed_train_df["immunogenicity"].apply(lambda x: 0 if x == "Negative" else 1).astype(int)
unprocessed_train_df["Label"].value_counts()

Label
0    4912
1    4059
Name: count, dtype: int64

In [12]:
#Remove the non-essential columns
dropped_train_df = unprocessed_train_df.drop(columns=["test","respond","potential","immunogenicity"],axis=1)
dropped_train_df.head(5)

#Map the HLA to HLA psuedosequence
dropped_train_df = standardize_hla_alleles(dropped_train_df)
dropped_train_df = map_alleles(dropped_train_df)
dropped_train_df.head(5)
print("Training data shape before: ",dropped_train_df.shape)

#Drop peptides with non-canonical amino acids
all_peptides_train = dropped_train_df["peptide"].tolist()
x_peptides_ids = [i for i in range(len(all_peptides_train)) if "X" in all_peptides_train[i]]

rev_dropped_train_df = dropped_train_df.drop(x_peptides_ids, axis=0 ).reset_index()
print("Training data shape after: ",rev_dropped_train_df.shape)

#Get the list of all peptides and create feature matrix
all_peptides_train = rev_dropped_train_df["peptide"].tolist()
all_peptides_features_df = feature_processing(all_peptides_train, seq_type="Peptide_")
print(all_peptides_features_df.shape)

all_hla_sequences = rev_dropped_train_df["hla_sequence"].tolist()
all_hla_features_df = feature_processing(all_hla_sequences, seq_type="HLA_")
print(all_hla_features_df.shape)

final_train_df = pd.concat([rev_dropped_train_df, all_peptides_features_df, all_hla_features_df], axis=1)
final_train_df.head(5)

final_train_df["Label"]=final_train_df["Label"].astype(int)
print(final_train_df["Label"].value_counts())
final_train_df.to_csv("../data/processed_training_data_with_features.csv",index=None)

Training data shape before:  (8971, 4)
Training data shape after:  (8969, 5)
Finished batch 0
Finished batch 5000
(8969, 111)
Finished batch 0
Finished batch 5000
(8969, 111)
Label
0    4912
1    4057
Name: count, dtype: int64


In [15]:
processed_train_df = pd.read_csv("../data/processed_training_data_with_features.csv",header="infer")
processed_train_df.head(5)

,index,peptide,HLA,Label,hla_sequence,Peptide_AF1,Peptide_AF2,Peptide_AF3,Peptide_AF4,Peptide_AF5,...,HLA_Z5,HLA_boman,HLA_hydrophobicity,HLA_charge,HLA_molecular_weight,HLA_aliphatic_index,HLA_instability_index,HLA_isoelectric_point,HLA_mz,HLA_structural_class
0,0,KEHVFFSEY,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.173955,-0.346010,0.376536,-0.138244,-0.186528,...,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906,alpha+beta
1,1,DEGATLYRF,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.130313,-0.141878,0.624118,0.427633,0.340208,...,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906,alpha+beta
2,2,TLAARIKFL,HLA-A*02:01,0,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,-0.236566,-0.667240,0.421759,0.748899,0.552450,...,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991,alpha+beta
3,3,KETLNEYKQL,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.644379,-0.419675,0.461561,0.160180,0.170256,...,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906,alpha+beta
4,4,STTDAEACY,HLA-A*01:01,0,YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY,-0.016626,-0.045450,-0.193642,0.402509,-0.348259,...,-0.141471,1.769118,-0.447059,0.085599,4259.78264,63.235294,-7.820588,7.501996,2129.501106,alpha+beta


### Preprocess Testing Data (preprocess --> extract features)